# AlphaBee Multi-Turn Dialogue Demo

演示 AlphaBee 多轮对话架构：每轮独立运行完整分析流水线，通过 LangChain `messages` 列表在多轮之间传递对话历史。

## 核心概念

- **无状态 LangGraph**：每轮创建新的图调用，不使用 checkpointer
- **外部历史记录**：`run_chat_session()` 用普通 Python 列表维护 `HumanMessage`/`AIMessage` 历史
- **全流水线重跑**：每轮依次执行 `collect_raw_facts → run_analysis_engines → run_thesis → review_thesis → generate_report → finalize_message`
- **子图流式**：`subgraphs=True` 捕获子 agent（如 FactCollectorAgent）的输出

In [1]:
import asyncio
import json
import sys
from typing import Any

from dotenv import load_dotenv
load_dotenv()

from langchain_core.messages import AIMessage, HumanMessage
from langfuse.langchain import CallbackHandler

from alphabee.orchestrator.agent import alphabee_agent
from alphabee.utils import get_logger

logger = get_logger("notebook")
print("✅ 环境初始化完成")

✅ 环境初始化完成


## 1. 模拟 run_chat_session 的简化版

这是 `main.py` 中 `run_chat_session` 的核心逻辑的精简版：
1. 将历史消息与当前问题合并为 `conversation` 列表
2. 调用 `alphabee_agent.astream()` 传入 `{"messages": conversation}`
3. 提取最终的 `AIMessage` 并追加到历史

In [7]:
async def run_single_turn(query: str, history: list[Any]) -> tuple[str, list[dict]]:
    """执行单轮查询，返回 (最终答案, 所有产物列表)"""
    conversation = [*history, HumanMessage(content=query)]
    final_answer = ""
    artifacts_by_type: dict[str, dict] = {}
    
    print(f"📨 发送 {len(conversation)} 条消息到 pipeline (含 {len(history)} 条历史)")
    
    async for namespace, chunk in alphabee_agent.astream(
        {"messages": conversation, "enhance": False, "llm_review": False},
        stream_mode="updates",
        subgraphs=True,
    ):
        for node_name, node_update in chunk.items():
            if not node_update:
                continue
            
            # 收集产物
            artifacts = node_update.get("artifacts", [])
            for a in artifacts:
                artifacts_by_type[a.type if hasattr(a, 'type') else 'unknown'] = (
                    a.model_dump(mode="json") if hasattr(a, 'model_dump') else a
                )
            
            # 提取 AIMessage
            messages = node_update.get("messages", [])
            for msg in messages:
                if isinstance(msg, AIMessage):
                    text = msg.content
                    if isinstance(text, list):
                        text = " ".join(
                            b.get("text", "") if isinstance(b, dict) else str(b)
                            for b in text
                        )
                    if text and len(text) > len(final_answer):
                        final_answer = text
    
    return final_answer, list(artifacts_by_type.values())

## 2. 解析产物 — 提取关键信息

最后一个 AIMessage 内容是一个 JSON，包含 `run`、`final_report`、`artifacts`、`decisions`、`issues`。

In [3]:
def parse_final_payload(answer: str) -> dict | None:
    """尝试将最终回答解析为 JSON 产物"""
    try:
        return json.loads(answer)
    except (json.JSONDecodeError, TypeError):
        return None

def summarize_payload(payload: dict) -> str:
    """简洁汇总产物"""
    run = payload.get("run", {})
    report = payload.get("final_report", {})
    artifacts = payload.get("artifacts", [])
    decisions = payload.get("decisions", [])
    issues = payload.get("issues", [])
    
    artifact_types = [a.get("type", "?") for a in artifacts]
    issue_severities = {}
    for i in issues:
        sev = i.get("severity", "unknown")
        issue_severities[sev] = issue_severities.get(sev, 0) + 1
    
    lines = [
        f"Run: {run.get('status', '?')} (goal: {run.get('goal', '?')[:40]}...)",
        f"Artifacts ({len(artifacts)}): {artifact_types}",
        f"Decisions: {len(decisions)}",
        f"Issues: {issue_severities}",
        f"Report keys: {list(report.keys()) if isinstance(report, dict) else 'non-dict'}",
    ]
    return "\n".join(lines)

## 3. 多轮对话演示

模拟 3 轮对话，每轮的问题都与前文相关，展示上下文如何贯穿。

In [4]:
async def demo_multi_turn():
    history: list[Any] = []
    queries = [
        "帮我分析一下贵州茅台（600519）的盈利能力",
        "它的毛利率和净利率具体是多少？",  # 依赖上一轮的"它"
        "和五粮液比怎么样？",            # 依赖前两轮上下文
    ]
    
    for turn, query in enumerate(queries, 1):
        print(f"{'='*60}")
        print(f"第 {turn} 轮 | 问题: {query}")
        print(f"{'='*60}")
        
        answer, artifacts = await run_single_turn(query, history)
        
        # 追加到历史
        history.append(HumanMessage(content=query))
        history.append(AIMessage(content=answer))
        
        # 解析并展示结果
        payload = parse_final_payload(answer)
        if payload:
            print(f"\n📊 产物汇总:\n{summarize_payload(payload)}")
            
            # 展示 report 摘要
            report = payload.get("final_report", {})
            if isinstance(report, dict) and "content" in report:
                content = report["content"]
                print(f"\n📝 报告摘要 (前 300 字):")
                print(content[:300] + "..." if len(content) > 300 else content)
        else:
            print(f"\n⚠ 回答无法解析为 JSON，原始文本 (前 200 字):\n{answer[:200]}...")
        
        print(f"\n📚 当前历史消息数: {len(history)}\n")
    
    print(f"\n{'='*60}")
    print("✅ 多轮对话完成")
    print(f"总共 {len(history)} 条消息 ({len(history)//2} 轮)")

await demo_multi_turn()

第 1 轮 | 问题: 帮我分析一下贵州茅台（600519）的盈利能力
📨 发送 1 条消息到 pipeline (含 0 条历史)
2026-06-22T11:00:23.724963 [info     ] langchain-openai detected https_proxy and no explicit `http_socket_options` / `http_client` / `http_async_client` / `openai_proxy`; skipping the custom `httpx` transport so httpx's env-proxy auto-detection applies. Pass `http_socket_options=[...]` to opt back into kernel-level TCP keepalive tuning on top of the env proxy.
messages length: 1
2026-06-22T11:00:31.115158 [info     ] llm.usage                      [alphabee.utils.llm] completion_reasoning_tokens=132 completion_tokens=293 component=agent.facts latency_ms=6635.72 model=deepseek-v4-pro prompt_cached_tokens=0 prompt_tokens=8751 provider=langchain_openai request_id=chatcmpl-80f0d0bc-3565-9bf7-8edc-9958a9102110 tags=['seq:step:1', 'component:agent.facts'] total_tokens=9044
messages length: 5
2026-06-22T11:01:42.386222 [info     ] llm.usage                      [alphabee.utils.llm] completion_reasoning_tokens=1796 completion_t

## 4. 手动交互式多轮对话

运行此 Cell 进入交互模式（需在支持 stdin 的环境中使用）。输入 `/clear` 清空上下文，输入 `/exit` 退出。

In [5]:
async def interactive_chat():
    """交互式多轮对话（适合 Terminal 中的 Jupyter）"""
    history: list[Any] = []
    turn = 1
    
    print("🐝 AlphaBee 交互式对话")
    print("   /clear — 清空上下文")
    print("   /exit  — 退出\n")
    
    while True:
        raw = input(f"[{turn:02d}] 你> ")
        query = raw.strip()
        
        if not query:
            continue
        if query.lower() in {"/exit", "/quit"}:
            print("👋 会话结束")
            break
        if query == "/clear":
            history.clear()
            turn = 1
            print("♻ 上下文已清空\n")
            continue
        
        answer, _ = await run_single_turn(query, history)
        history.append(HumanMessage(content=query))
        history.append(AIMessage(content=answer))
        
        # 打印 report 内容
        payload = parse_final_payload(answer)
        if payload:
            report = payload.get("final_report", {})
            if isinstance(report, dict):
                content = report.get("content", str(report))
            else:
                content = str(report)
            print(f"\n{content[:1500]}\n")
            if len(content) > 1500:
                print(f"...(已截断，完整 {len(content)} 字)\n")
        else:
            print(f"\n{answer[:1500]}\n")
        
        turn += 1

# 取消注释运行（在 Jupyter notebook 中 input() 可能无法正常工作）:
# await interactive_chat()

## 5. 查看 Pipeline 内部状态

演示如何访问每轮运行后的内部状态：`OrchestratorState` 包含 `run`、`steps`、`artifacts`、`decisions`、`issues`。

In [6]:
async def demo_state_inspection():
    """查看单次运行后的完整状态"""
    query = "分析一下比亚迪的估值水平"
    conversation = [HumanMessage(content=query)]
    
    # 收集所有 node 输出
    all_state_updates: list[dict] = []
    
    async for namespace, chunk in alphabee_agent.astream(
        {"messages": conversation, "enhance": False, "llm_review": False},
        stream_mode="updates",
        subgraphs=True,
    ):
        for node_name, node_update in chunk.items():
            if node_update:
                all_state_updates.append({
                    "namespace": namespace,
                    "node": node_name,
                    "keys": list(node_update.keys()),
                })
    
    print(f"总共 {len(all_state_updates)} 次状态更新\n")
    for i, upd in enumerate(all_state_updates):
        ns_path = " > ".join(
            n.replace(":", ": ") for n in upd["namespace"]
        ) if upd["namespace"] else "ROOT"
        print(f"[{i:02d}] [{ns_path}] node={upd['node']}")
        print(f"     keys: {upd['keys']}")

await demo_state_inspection()

messages length: 1
2026-06-22T11:04:43.283960 [info     ] llm.usage                      [alphabee.utils.llm] completion_reasoning_tokens=128 completion_tokens=177 component=agent.facts latency_ms=5139.55 model=deepseek-v4-pro prompt_cached_tokens=8704 prompt_tokens=8746 provider=langchain_openai request_id=chatcmpl-956566a8-64d9-9771-ae91-5458468463eb tags=['seq:step:1', 'component:agent.facts'] total_tokens=8923
messages length: 3
2026-06-22T11:04:50.245017 [info     ] llm.usage                      [alphabee.utils.llm] completion_reasoning_tokens=119 completion_tokens=316 component=agent.facts latency_ms=6672.44 model=deepseek-v4-pro prompt_cached_tokens=8704 prompt_tokens=8815 provider=langchain_openai request_id=chatcmpl-858067c0-0352-9279-a00b-7341a0aaaa71 tags=['seq:step:1', 'component:agent.facts'] total_tokens=9131
2026-06-22T11:04:50.556201 [warning  ] Tushare API error, retrying    [alphabee.collectors.tushare.helper] api=sw_daily attempt=1 error='抱歉，您没有接口(sw_daily)访问权限，权限的具

## 6. 架构说明

### 关键文件

| 文件 | 职责 |
|------|------|
| `main.py:652` | `run_chat_session()` — CLI 多轮循环 |
| `main.py:446` | `run_query()` — 单轮执行 + 流式渲染 |
| `main.py:440` | `_append_turn_history()` — 追加 Human/AI 消息对 |
| `alphabee/orchestrator/agent.py:252` | `alphabee_agent` — 编译后的 LangGraph StateGraph |
| `alphabee/orchestrator/state.py` | `OrchestratorState` — 共享 TypedDict 状态 |
| `alphabee/core/schemas.py` | `Run`, `Step`, `Artifact`, `Decision`, `Issue` — 数据模型 |

### 消息流

```
User input "查询3"
  │
  ▼
conversation = [HumanMessage("查询1"), AIMessage("答案1"),
                HumanMessage("查询2"), AIMessage("答案2"),
                HumanMessage("查询3")]
  │
  ▼
alphabee_agent.astream({"messages": conversation})
  │
  ├─ collect_raw_facts      (FactCollectorAgent + 结构化提取)
  ├─ run_analysis_engines   (DerivedFacts + Signal + Anomaly)
  ├─ run_thesis             (ThesisEngine ± LLM增强)
  ├─ review_thesis          (ThesisReviewer)
  ├─ generate_report        (LLM → Markdown)
  └─ finalize_message       (合并 → JSON AIMessage)
  │
  ▼
history.append(HumanMessage("查询3"))
history.append(AIMessage(final_json_answer))
```

### 依赖注入

历史上下文通过 `messages` 字段传入 LangGraph state：

```python
conversation = [*history, HumanMessage(content=current_query)]
agent.astream({"messages": conversation, ...})
```

子 agent（如 `FactCollectorAgent`）通过 LangGraph 的 `subgraphs=True` 机制自动接收父图的 `MessagesState`。